### Solutions for 02-Working_with_ASDF_Files

**Exercise 1:**

In [1]:
import asdf

**Read the file and use the `info` method to look at the contents**.

`info` has arguments which control the behavior. Use the defaults to look at the contents of the file.

In [2]:
af = asdf.open('../data/r0000101001001001001_0001_wfi01_f158_cal.asdf')

In [3]:
af.info(max_rows=200)

root (AsdfObject)
├─asdf_library (Software)
│ ├─author (str): The ASDF Developers
│ ├─homepage (str): http://github.com/asdf-format/asdf
│ ├─name (str): asdf
│ └─version (str): 5.3.1
├─history (dict)
│ └─extensions (list)
│   ├─[0] (ExtensionMetadata)
│   │ ├─extension_class (str): asdf.extension._manifest.ManifestExtension
│   │ ├─extension_uri (str): asdf://astropy.org/astropy/extensions/units-1.3.0
│   │ └─software (Software) ...
│   ├─[1] (ExtensionMetadata)
│   │ ├─extension_class (str): asdf.extension._manifest.ManifestExtension
│   │ ├─extension_uri (str): asdf://asdf-format.org/astronomy/extensions/astronomy-1.2.0
│   │ ├─manifest_software (Software) ...
│   │ └─software (Software) ...
│   ├─[2] (ExtensionMetadata)
│   │ ├─extension_class (str): asdf.extension._manifest.ManifestExtension
│   │ ├─extension_uri (str): asdf://stsci.edu/datamodels/roman/extensions/datamodels-2.1.0
│   │ ├─manifest_software (Software) ...
│   │ └─software (Software) ...
│   ├─[3] (ExtensionMetadata)

In [4]:
af.search('wcs')

root (AsdfObject)
└─roman (WfiImage) # Level 2 (L2) Calibrated Roman Wide Field Instrument (WFI) Rate Image.
  └─meta (dict)
    ├─cal_step (dict) # Level 2 Calibration Status
    │ └─assign_wcs (str): COMPLETE
    ├─wcs (WCS) # WCS object
    ├─wcs_fit_results (dict): {'<rot>': 3.3946225943483266e-05, '<scale>': 1.0000011396529942, 'center': [-3 (truncated)
    └─wcsinfo (dict): {'aperture_name': 'WFI01_FULL', 'dec_ref': 66.0355095513252, 'ra_ref': 269.83252548557 (truncated)

In [5]:
w = af['roman']['meta']['wcs']
print(w)

   From                       Transform                   
---------- -----------------------------------------------
  detector                                   CompoundModel
      v2v3                                  DVA_Correction
v2v3vacorr Roman V2-V3 tangent-plane linear correction. v1
  v2v3corr                                        v23tosky
     world                                            None


In [6]:
ra, dec = w(200, 300)
print(ra, dec)

269.9718550194006 65.98323705547408


In [7]:
sky = w.pixel_to_world(200, 300)
print(sky)

<SkyCoord (ICRS): (ra, dec) in deg
    (269.97185502, 65.98323706)>


In [8]:
af['roman']['data']

array([[0.3003652 , 0.17354345, 0.24824238, ..., 0.28168783, 0.1751165 ,
        0.28870606],
       [0.21080498, 0.21472143, 0.19666195, ..., 0.2118787 , 0.23360458,
        0.18349892],
       [0.21772373, 0.19066802, 0.15711647, ..., 0.17154558, 0.21344924,
        0.22230656],
       ...,
       [0.14813541, 0.24864507, 0.25581175, ...,        nan, 0.24368694,
               nan],
       [0.22111104, 0.22793521, 0.1688724 , ...,        nan, 0.23213342,
               nan],
       [0.2526113 , 0.11470476, 0.24369204, ...,        nan, 0.12508896,
               nan]], shape=(4088, 4088), dtype=float32)

In [9]:
af['roman']['data'][0, 0] = 999

In [10]:
af['roman']['data']

array([[9.9900000e+02, 1.7354345e-01, 2.4824238e-01, ..., 2.8168783e-01,
        1.7511649e-01, 2.8870606e-01],
       [2.1080498e-01, 2.1472143e-01, 1.9666195e-01, ..., 2.1187870e-01,
        2.3360458e-01, 1.8349892e-01],
       [2.1772373e-01, 1.9066802e-01, 1.5711647e-01, ..., 1.7154558e-01,
        2.1344924e-01, 2.2230656e-01],
       ...,
       [1.4813541e-01, 2.4864507e-01, 2.5581175e-01, ...,           nan,
        2.4368694e-01,           nan],
       [2.2111104e-01, 2.2793521e-01, 1.6887240e-01, ...,           nan,
        2.3213342e-01,           nan],
       [2.5261131e-01, 1.1470476e-01, 2.4369204e-01, ...,           nan,
        1.2508896e-01,           nan]], shape=(4088, 4088), dtype=float32)

**Exercise 2:**

Add `additionalProperties=false` to the schema and open the file

In [11]:
s = """
%YAML 1.1
---
$schema: http://stsci.edu/schemas/yaml-schema/draft-01

title: Mickey's pet
description: |
  Basic info and a picture of Mickie's 
  dog Pluto.

type: object
properties:
  age:
    title: The age of Pluto
    type: object
    properties:
      birthday:
        title: Pluto's first showing
        tag: tag:stsci.edu:asdf/time/time-1.1.0
  mass:
    title: How much he weighs.
    tag: tag:stsci.edu:asdf/unit/quantity-1.1.0
  picture:
    tag: tag:stsci.edu:asdf/core/ndarray-1.0.0
  name:
    title: Name
    type: string
required: [name, picture]
additionalProperties: false
...
"""

In [12]:
f = open('add-prop-1.0.0.yaml', mode='w')
f.write(s)
f.close()

In [13]:
# Expected validation error
afs = asdf.open('pluto.asdf', custom_schema='./add-prop-1.0.0.yaml')

/opt/anaconda3/envs/roman-data-workshop-env/lib/python3.14/site-packages/asdf/_asdf.py:1545: AsdfFutureWarning: In the future validation on read will be off by default. Set AsdfConfig.validate_on_read to False to opt-in to the new behavior, or to True to silence this warning.
  warnings.warn(


ValidationError: Additional properties are not allowed ('asdf_library', 'birthday', 'history' were unexpected)

Failed validating 'additionalProperties' in schema:
    {'$schema': 'http://stsci.edu/schemas/yaml-schema/draft-01',
     'additionalProperties': False,
     'description': "Basic info and a picture of Mickie's \ndog Pluto.\n",
     'properties': {'age': {'properties': {'birthday': {'tag': 'tag:stsci.edu:asdf/time/time-1.1.0',
                                                        'title': "Pluto's "
                                                                 'first '
                                                                 'showing'}},
                            'title': 'The age of Pluto',
                            'type': 'object'},
                    'mass': {'tag': 'tag:stsci.edu:asdf/unit/quantity-1.1.0',
                             'title': 'How much he weighs.'},
                    'name': {'title': 'Name', 'type': 'string'},
                    'p

Take the original schema and add a new required property, called `friend`.

In [14]:
s = """
%YAML 1.1
---
$schema: http://stsci.edu/schemas/yaml-schema/draft-01

title: Mickey's pet
description: |
  Basic info and a picture of Mickie's 
  dog Pluto.

type: object
properties:
  age:
    title: The age of Pluto
    type: object
    properties:
      birthday:
        title: Pluto's first showing
        tag: tag:stsci.edu:asdf/time/time-1.1.0
  mass:
    title: How much he weighs.
    tag: tag:stsci.edu:asdf/unit/quantity-1.1.0
  picture:
    tag: tag:stsci.edu:asdf/core/ndarray-1.0.0
  name:
    title: Name
    type: string
  friend:
    type: string
    title: "Who is Pluto's friend?"
required: [name, picture, friend]
...
"""

In [15]:
f = open('pluto-friend-1.0.0.yaml', mode='w')
f.write(s)
f.close()

In [16]:
# Expected validation error
afs = asdf.open('pluto.asdf', custom_schema='./pluto-friend-1.0.0.yaml')

ValidationError: 'friend' is a required property

Failed validating 'required' in schema:
    {'$schema': 'http://stsci.edu/schemas/yaml-schema/draft-01',
     'description': "Basic info and a picture of Mickie's \ndog Pluto.\n",
     'properties': {'age': {'properties': {'birthday': {'tag': 'tag:stsci.edu:asdf/time/time-1.1.0',
                                                        'title': "Pluto's "
                                                                 'first '
                                                                 'showing'}},
                            'title': 'The age of Pluto',
                            'type': 'object'},
                    'friend': {'title': "Who is Pluto's friend?",
                               'type': 'string'},
                    'mass': {'tag': 'tag:stsci.edu:asdf/unit/quantity-1.1.0',
                             'title': 'How much he weighs.'},
                    'name': {'title': 'Name', 'type': 'string'},
                

In [17]:
asf = asdf.open('pluto.asdf', mode='rw')
asf['friend'] = 'Mickey'

asf.write_to('pluto-friend.asdf')

In [18]:
asf = asdf.open('pluto-friend.asdf', custom_schema='./pluto-friend-1.0.0.yaml')